# 01 · Las cinco viviendas más alejadas de la mediana por costa

Práctica basada en `notebooks/01_viviendas.ipynb`. Tiempo sugerido: 20–30 minutos.

**Tu tarea:** completá únicamente la función de la sección 3.
La carga de datos, el entorno y las pruebas ya están preparados.

1. Elegí el kernel del entorno `.venv` del proyecto y ejecutá las celdas de preparación.
2. Leé el contrato y reemplazá el `raise NotImplementedError` por tu implementación.
3. Ejecutá de nuevo la celda de la función y después la de pruebas.
4. Si todas pasan, esa misma celda probará tu función sobre los datos reales y mostrará el resultado.

Podés usar **Ejecutar todo** desde un kernel nuevo. Mientras la función esté pendiente,
la última celda mostrará un aviso; no tenés que instalar nada adicional en el entorno actual.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `../datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Retomá el livecoding anterior: para cada `ocean_proximity`, calculá la mediana de
`median_house_value` y extraé las **cinco filas cuya distancia absoluta a esa mediana sea mayor**.
Recordá que cada fila representa una zona censal.

- Usá todas las filas de la base preparada, incluidos los precios en el máximo observado.
- Devolvé las columnas originales, seguidas por `mediana_grupo` y `distancia_mediana`.
- Conservá el índice original. Ordená las categorías alfabéticamente y, dentro de cada una,
  la distancia de mayor a menor. Si hay empate, conservá el orden original de las filas.
- `n` es un entero positivo. Si un grupo tiene menos filas, devolvé todas las disponibles.
- Una entrada vacía debe devolver un DataFrame vacío con las columnas indicadas.
- No modifiques el DataFrame recibido.

**Ejemplo:** precios `[100, 200, 300, 400, 1000]` → mediana `300` → distancias
`[200, 100, 0, 100, 700]`. Con `n=2`, se extraen las filas de `1000` y `100`.

## 3. Tu función

Editá solo esta celda. Mantené el nombre de la función y sus parámetros para que funcionen las pruebas.

In [3]:
def extraer_extremos_por_costa(datos, n=5):
    """Implementá el contrato de la sección 2 sin modificar datos."""
    # TODO: escribí tu solución acá.
    raise NotImplementedError("Completá esta función y volvé a ejecutar las pruebas.")

## 4. Pruebas y ejecución real

Un `AssertionError` indica qué resultado difiere del esperado. Primero revisá el caso pequeño que falló. Después de editar la función, ejecutá su celda antes de volver a correr estas pruebas.

Las pruebas verifican ejemplos pequeños y propiedades del resultado real; no sustituyen una revisión de tu razonamiento.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "ocean_proximity": ["B", "A", "A", "B", "A", "C", "B"],
        "median_house_value": [10, 100, 0, 20, 50, 7, 30],
        "identificador": list("abcdefg"),
    }, index=[90, 7, 42, 3, 8, 55, 1])
    original = caso.copy(deep=True)

    # Empates, grupos con menos de n filas y conservación de las columnas originales.
    esperado = caso.iloc[[1, 2, 0, 6, 5]].copy()
    esperado["mediana_grupo"] = [50, 50, 20, 20, 7]
    esperado["distancia_mediana"] = [50, 50, 10, 10, 0]
    comparar(funcion(caso, n=2), esperado)
    comparar(caso, original)

    comparar(funcion(caso, n=1), esperado.iloc[[0, 2, 4]])

    esperado_todos = caso.iloc[[1, 2, 4, 0, 6, 3, 5]].copy()
    esperado_todos["mediana_grupo"] = [50, 50, 50, 20, 20, 20, 7]
    esperado_todos["distancia_mediana"] = [50, 50, 0, 10, 10, 0, 0]
    comparar(funcion(caso, n=99), esperado_todos)

    vacio = caso.iloc[:0].copy()
    esperado_vacio = vacio.copy()
    esperado_vacio["mediana_grupo"] = pd.Series(dtype=float)
    esperado_vacio["distancia_mediana"] = pd.Series(dtype=float)
    comparar(funcion(vacio, n=5), esperado_vacio)
    comparar(caso, original)
    print("OK: extremos, empates, n variable, grupo pequeño, entrada vacía y datos intactos.")


resultado = None
try:
    probar(extraer_extremos_por_costa)
except NotImplementedError:
    print("PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.")
else:
    # Comprobamos también que el dataset real permanezca intacto.
    antes = datos.copy(deep=True)
    resultado = extraer_extremos_por_costa(datos)
    comparar(datos, antes)
    assert list(resultado.columns) == list(datos.columns) + ["mediana_grupo", "distancia_mediana"]
    assert len(resultado) == datos.groupby("ocean_proximity").size().clip(upper=5).sum()
    for grupo, filas in resultado.groupby("ocean_proximity"):
        valores = datos.loc[datos["ocean_proximity"] == grupo, "median_house_value"]
        distancias = np.sort(np.abs(valores.to_numpy() - np.median(valores)))[::-1][:5]
        np.testing.assert_allclose(filas["distancia_mediana"], distancias)
        np.testing.assert_allclose(filas["mediana_grupo"], np.median(valores))
    print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
    display(resultado.head(20))

PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.


## Pista opcional

<details>
<summary>Abrir solo si te trabás</summary>

`groupby`, `transform`, `median`, `abs`, `sort_values` con orden estable y `head` por grupo.

</details>

**Para explicar en voz alta:** ¿qué filas entran al cálculo?, ¿qué representa cada fila de la salida?, ¿qué caso límite cubre cada prueba?